## Vehicle Damage Detection Project: Hyperparameter Tunning

In [1]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [2]:
device = ("cuda" if torch.cuda.is_available()
          else "mps" 
          if torch.backends.mps.is_available()
          else "cpu")
device

'mps'

## Load Data

In [3]:
# Define the image transformations for preprocessing and augmentation
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
dataset_path = "./dataset"

dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms)
len(dataset)

2300

In [5]:
dataset.classes

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [6]:
num_classes = len(dataset.classes)
num_classes

6

In [7]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size

train_size, val_size

(1725, 575)

In [8]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

## Model Training & Hyperparameter Tunning

In [10]:
# Load the pre-trained ResNet model
class CarClassifierResNet(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.model = models.resnet50(weights='DEFAULT')
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False
            
        # Unfreeze layer4 and fc layers
        for param in self.model.layer4.parameters():
            param.requires_grad = True            
            
        # Replace the final fully connected layer
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features, num_classes)
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [11]:
# Define the objective function for Optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    
    # Load the model
    model = CarClassifierResNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)
    
    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    # Training loop (using fewer epochs for faster hyperparameter tuning)
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        
        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)
        
        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time: {end - start} seconds")
    
    return accuracy

In [12]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-07-15 11:11:41,166] A new study created in memory with name: no-name-e3fe8583-1f4e-4d47-8f44-18ac7b915aee
[I 2026-07-15 11:14:20,474] Trial 0 finished with value: 75.47826086956522 and parameters: {'lr': 0.002035523113246686, 'dropout_rate': 0.29099927235020023}. Best is trial 0 with value: 75.47826086956522.


Execution time: 158.7203061580658 seconds


[I 2026-07-15 11:17:04,929] Trial 1 finished with value: 72.34782608695652 and parameters: {'lr': 0.005411687435202689, 'dropout_rate': 0.6379334872639421}. Best is trial 0 with value: 75.47826086956522.


Execution time: 163.90358209609985 seconds


[I 2026-07-15 11:20:32,523] Trial 2 finished with value: 72.52173913043478 and parameters: {'lr': 0.00022063089699461108, 'dropout_rate': 0.547643791419768}. Best is trial 0 with value: 75.47826086956522.


...

[I 2026-07-15 11:58:18,465] Trial 14 pruned. 
[I 2026-07-15 12:03:03,211] Trial 15 finished with value: 76.69565217391305 and parameters: {'lr': 0.002663847104377559, 'dropout_rate': 0.2717495069414879}. Best is trial 10 with value: 77.04347826086956.


Execution time: 282.0333640575409 seconds


[I 2026-07-15 12:08:08,621] Trial 16 finished with value: 76.34782608695652 and parameters: {'lr': 0.0003232395845654328, 'dropout_rate': 0.25036345399228355}. Best is trial 10 with value: 77.04347826086956.


Execution time: 304.6464750766754 seconds


[I 2026-07-15 12:11:42,158] Trial 17 pruned. 
[I 2026-07-15 12:15:23,310] Trial 18 pruned. 
[I 2026-07-15 12:17:02,189] Trial 19 pruned. 


## Print best parameters

In [14]:
study.best_params

{'lr': 0.0009759645600462505, 'dropout_rate': 0.3487833467617113}